# Test d'une courbe de sertissage avec One-Class SVM
Ce notebook charge l'artefact sauvegardé (`crimp_one_class_svm_model.pkl` contenant le modèle et le `StandardScaler`), normalise les caractéristiques d'une nouvelle courbe et détecte s'il s'agit d'une pièce conforme (**OK**) ou d'une anomalie (**NOK**).

## 1. Importation des bibliothèques

In [1]:
import joblib
import numpy as np
import pandas as pd

## 2. Chargement du modèle One-Class SVM et du Scaler

In [5]:
artifact = joblib.load("./../models/crimp_one_class_svm_model.pkl")
ocsvm_model = artifact['model']
scaler = artifact['scaler']
print("Modèle One-Class SVM et Scaler chargés avec succès.")

Modèle One-Class SVM et Scaler chargés avec succès.


## 3. Définition ou chargement d'une nouvelle courbe brute

In [6]:
# Exemple de courbe brute (remplacer par les valeurs d'une acquisition capteur réelle)
nouvelle_courbe_brute = [12.5, 14.2, 18.0, 25.6, 45.1, 78.3, 95.0, 85.2, 50.4, 30.1, 15.0]
print(f"Nombre de points dans la courbe : {len(nouvelle_courbe_brute)}")

Nombre de points dans la courbe : 11


## 4. Extraction et normalisation des caractéristiques

In [7]:
arr = np.array(nouvelle_courbe_brute)
features = [
    np.max(arr),       # Force maximale atteinte
    np.min(arr),       # Force minimale
    np.mean(arr),      # Force moyenne
    np.std(arr),       # Écart-type (variation du signal)
    np.trapz(arr),     # Aire sous la courbe (énergie de sertissage)
    np.argmax(arr)     # Position temporelle du pic de force
]

X_input = np.array([features])

# Normalisation obligatoire avec le scaler entraîné
X_input_scaled = scaler.transform(X_input)
print("Caractéristiques normalisées :", X_input_scaled)

Caractéristiques normalisées : [[-97.9568241  -66.82296182 -97.2198175  -18.38109095 -97.44724998
  -11.7652187 ]]


## 5. Prédiction et interprétation du statut

In [8]:
# One-Class SVM retourne 1 pour OK et -1 pour une anomalie
raw_pred = ocsvm_model.predict(X_input_scaled)[0]

# Conversion au format standard (1 -> 0 pour OK, -1 -> 1 pour NOK)
statut = 0 if raw_pred == 1 else 1

print("--- Résultat du test (One-Class SVM) ---")
print(f"Statut prédit : {'OK (Conforme)' if statut == 0 else 'NOK (Anomalie)'}")

--- Résultat du test (One-Class SVM) ---
Statut prédit : NOK (Anomalie)
